In [ ]:
#Cell 1 — import
from src.protogen_preprocessing import (
    load_protogen_data,
    filter_annotation_ra_bl,
    select_relevant_measurement_columns,
    add_marker_name,
    transpose_measurement,
    merge_with_annotation,
    split_metadata_features,
    inspect_missing_values,
)
import pandas as pd

In [ ]:
#Cell 2 — load
data = load_protogen_data()

df_measurement = data["df_measurement"]
df_annotation = data["df_annotation"]

In [ ]:
#Cell 3 — inspect annotation quickly
print(df_annotation.columns.tolist())
df_annotation.head()

In [ ]:
#Cell 4 — filter to TACERA + BL
df_annotation_ra_bl = filter_annotation_ra_bl(df_annotation)
df_annotation_ra_bl.head()

In [ ]:
#Cell 5 — sanity check
print(df_annotation_ra_bl["Study"].value_counts(dropna=False))
print(df_annotation_ra_bl["Timepoint"].value_counts(dropna=False))

In [ ]:
#Cell 6 - selecting relevant sample columns
#Takes fixed metadata columns from the measurement sheet. It keeps: ProteinID, GeneID, Gene Symbol, Gene Name
#Takes the relevant SampleIds from the filtered annotation table and Keeps only those matching sample columns in the measurement sheet
df_measurement_ra_bl = select_relevant_measurement_columns(df_measurement, df_annotation_ra_bl)
df_measurement_ra_bl.head()
print(df_measurement_ra_bl.shape)

In [ ]:
#Cell 7 - Create a unique marker name, MarkerName = Gene Symbol + "_" + ProteinID
#Right now your rows are markers, and later after transpose these marker names become the column names.
#We need it so that after the transpose, every future feature column has a clean, unique name
df_measurement_ra_bl = add_marker_name(df_measurement_ra_bl)
df_measurement_ra_bl[["ProteinID", "Gene Symbol", "MarkerName"]].head()

In [ ]:
#Cell 8 - Transpose the measurement dataset so that rows become columns and columns become rows. This way, every row is a sample, and every column is a marker/feature. This is the format we need for machine learning.
df_measurement_t = transpose_measurement(df_measurement_ra_bl)
df_measurement_t.head()
print(df_measurement_t.shape)

In [ ]:
#Cell 9 - Mering the transposed measurement table with the filtered annotation table using SampleId.
df_gene_merged = merge_with_annotation(df_measurement_t, df_annotation_ra_bl)
df_gene_merged.head()
print(df_gene_merged.shape)

In [ ]:
#Cell 10 - Split metadata columns and feature columns since we want to do the cleaning on the feature columns only.
df_meta, df_features = split_metadata_features(df_gene_merged)

df_meta.head()
print(df_meta.shape)
print(df_features.shape)

In [ ]:
#Cell 10 - Sanity check on df_meta
print("Metadata shape:", df_meta.shape)
display(df_meta.head())

print("\nUnique SampleIds:", df_meta["SampleId"].nunique())
print("Unique Patient_ID:", df_meta["Patient_ID"].nunique())
print("Unique Digest:", df_meta["Digest"].nunique())

print("\nStudy values:")
print(df_meta["Study"].value_counts(dropna=False))

print("\nTimepoint values:")
print(df_meta["Timepoint"].value_counts(dropna=False))

In [ ]:
#Cell 11 - Sanity check on df_features
print("Feature shape:", df_features.shape)
display(df_features.head())

print("\nDtypes summary:")
print(df_features.dtypes.value_counts())

print("\nBasic stats:")
display(df_features.describe().T.head(10))

In [ ]:
# Cell 12 - inspect missing values
missing_summary = inspect_missing_values(df_features)

display(missing_summary.head(10))
print("Total missing values in df_features:", df_features.isna().sum().sum())